### Definicoes (FedAvgCNN, ataques, normalizacao)

In [ ]:
import copy
import torch
from torch import nn
from datasets import Dataset

PAD_ID = 0
NUM_BINS = 10000
MAX_LENGTH = 512
ATTACK_TYPES = ['zeros', 'random', 'shuffle', 'noise']


class FedAvgCNN(nn.Module):
    def __init__(self, in_features=1, num_classes=10, dim=1024):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_features, 32, kernel_size=5, padding=0, stride=1, bias=True),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 2)),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=5, padding=0, stride=1, bias=True),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 2)),
        )
        self.fc1 = nn.Sequential(nn.Linear(dim, 512), nn.ReLU(inplace=True))
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        out = torch.flatten(out, 1)
        out = self.fc1(out)
        out = self.fc(out)
        return out


# ----- Ataques originais (versao "easy") -----
def model_zeros(model, device='cpu'):
    copy_model = copy.deepcopy(model)
    for param in copy_model.parameters():
        param.data.zero_()
    return copy_model.to(device)


def random_param(model, device='cpu'):
    copy_model = copy.deepcopy(model)
    for param in copy_model.parameters():
        param.data = torch.rand_like(param.data).to(device)
    return copy_model


def shuffle_model(model, device='cpu'):
    copy_model = copy.deepcopy(model)
    for param in copy_model.parameters():
        data_flatten = param.data.view(-1)
        index_random = torch.randperm(len(data_flatten))
        shuffled_param = data_flatten[index_random]
        param.data = shuffled_param.view(param.shape).to(device)
    return copy_model


def model_noise(model, SNR=10.0, device='cpu'):
    copy_model = copy.deepcopy(model)
    for param in copy_model.parameters():
        with torch.no_grad():
            power_signal = torch.mean(param.data ** 2)
            if power_signal == 0:
                continue
            power_noise = power_signal / (10 ** (SNR / 10))
            noise = torch.normal(0.0, torch.sqrt(power_noise), size=param.shape).to(device)
            param.data.add_(noise)
    return copy_model


def create_benign_model(device='cpu'):
    """Simula 1 rodada local: copia o modelo global (pretrained ou random) +
    ruido pequeno como proxy de 1 step de SGD em data shard local.
    Pressupoe `pretrained_base` definida (vide cell de config)."""
    model = copy.deepcopy(pretrained_base).to(device)
    for param in model.parameters():
        param.data.add_(torch.normal(0.0, 0.01, size=param.shape).to(device))
    return model


def create_malicious_model(base_model, attack_type='random', device='cpu'):
    """Ataque ORIGINAL (easy). Usado quando HARDEN_ATTACKS=False."""
    if attack_type == 'zeros':
        return model_zeros(base_model, device)
    if attack_type == 'random':
        return random_param(base_model, device)
    if attack_type == 'shuffle':
        return shuffle_model(base_model, device)
    if attack_type == 'noise':
        return model_noise(base_model, SNR=5.0, device=device)
    raise ValueError("Ataque invalido")


def preprocess_weights(state_dict, max_length=MAX_LENGTH, num_bins=NUM_BINS):
    """Pooling estratificado per-camada (linspace) + bins. Usado no
    in-memory dataset (nao usado pelo detector.py, que tem sua propria
    versao com normalizacao por quantis)."""
    parts = [v.flatten() for k, v in state_dict.items() if 'weight' in k]
    weights = torch.cat(parts)
    n = len(weights)
    weights_norm = (weights - weights.min()) / (weights.max() - weights.min() + 1e-8)
    if n >= max_length:
        idx = torch.linspace(0, n - 1, steps=max_length).long()
        sampled = weights_norm[idx]
    else:
        sampled = weights_norm
    binned = (sampled * (num_bins - 1)).long() + 1
    if len(binned) < max_length:
        pad = torch.full((max_length - len(binned),), PAD_ID, dtype=torch.long)
        binned = torch.cat([binned, pad])
    return binned.tolist()


def tokenize_function(examples):
    return {
        'input_ids': examples['inputs'],
        'attention_mask': [[1 if tok != PAD_ID else 0 for tok in ids] for ids in examples['inputs']],
    }


### Grid de variantes do dataset

Quatro datasets ortogonais controlados por dois flags:

| variante | USE_PRETRAINED_BASE | HARDEN_ATTACKS | descricao |
|---|---|---|---|
| 1. **Leakage**          | `False` | `False` | base_model unico compartilhado + ataques originais (U[0,1], full shuffle, SNR fixo). Equivalente ao dataset inicial. |
| 2. **Hard**             | `False` | `True`  | random init + ataques sutis (smart_random Gaussian com sigma da camada, shuffle parcial, SNR variavel). |
| 3. **Pretrained+Hard**  | `True`  | `True`  | FedAvgCNN treinada em MNIST + ataques sutis. Mais realista. |
| 4. **Pretrained+Easy**  | `True`  | `False` | FedAvgCNN treinada em MNIST + ataques originais. Isola o efeito de pretreinar. |

Edite os flags abaixo, rode esta celula (treina ou carrega `pretrained_base`), depois rode a proxima (gera `state_dicts/`). Final: `python detector.py` (DistilBERT) ou `python detector_mlp.py` (MLP).


In [ ]:
# ===== CONFIG =====
USE_PRETRAINED_BASE = True   # True: treina FedAvgCNN em MNIST; False: usa random init
HARDEN_ATTACKS = True        # True: ataques sutis e fresh base por amostra; False: ataques originais com base compartilhada
N_SAMPLES_PER_CLASS = 1000   # tamanho do dataset (200 para iteracao rapida, 1000 para metricas estaveis)


def train_base_model(epochs=10, batch_size=128, lr=0.01, device=None):
    """Treina FedAvgCNN em MNIST. Da estrutura espacial aos pesos para que
    shuffle se torne detectavel pelas features espectrais e espaciais."""
    import torchvision
    from torchvision import transforms
    from torch.utils.data import DataLoader

    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    transform = transforms.Compose([transforms.ToTensor()])
    train_ds = torchvision.datasets.MNIST(root='./mnist_data', train=True, download=True, transform=transform)
    test_ds = torchvision.datasets.MNIST(root='./mnist_data', train=False, download=True, transform=transform)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=2)

    model = FedAvgCNN(in_features=1, num_classes=10, dim=1024).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    for ep in range(epochs):
        model.train()
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            running += loss.item()
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb).argmax(dim=1)
                correct += (pred == yb).sum().item()
                total += yb.numel()
        print(f'  epoch {ep+1:2d}/{epochs} loss={running/len(train_loader):.4f} test_acc={correct/total:.4f}')
    return model.cpu()


# Setup do pretrained_base segundo a flag
if USE_PRETRAINED_BASE:
    print('USE_PRETRAINED_BASE=True -> treinando FedAvgCNN em MNIST...')
    torch.manual_seed(0)
    pretrained_base = train_base_model(epochs=10)
    print('pretrained_base pronto (treinado).')
else:
    print('USE_PRETRAINED_BASE=False -> random init.')
    torch.manual_seed(42)
    pretrained_base = FedAvgCNN()
    print('pretrained_base pronto (random init).')

# base_model e alias usado pelos ataques no modo nao-hardened (mantem compatibilidade)
base_model = pretrained_base


### Salvamento json e tensors

In [ ]:
import copy
import json
import os
import torch
from safetensors.torch import save_file


def _make_fresh_base():
    """Base do malicioso = copia do pretrained_base + ruido pequeno (fresh por amostra).
    Anti-leakage: cada malicioso tem ruido independente."""
    m = copy.deepcopy(pretrained_base)
    for p in m.parameters():
        p.data.add_(torch.normal(0.0, 0.01, size=p.shape))
    return m


os.makedirs('state_dicts', exist_ok=True)
state_dicts_list = []

for i in range(N_SAMPLES_PER_CLASS):
    # Benigno: sempre via create_benign_model (usa pretrained_base)
    torch.manual_seed(1000 + i)
    benign_sd = create_benign_model().state_dict()
    state_dicts_list.append({
        'state_dict': benign_sd,
        'label': 0,
        'type': 'benign',
        'id': f'benign_{i:04d}',
    })

    # Malicioso: condicional ao flag HARDEN_ATTACKS
    torch.manual_seed(2000 + i)
    attack = ATTACK_TYPES[i % len(ATTACK_TYPES)]

    if HARDEN_ATTACKS:
        # === Ataques endurecidos ===
        fresh_base = _make_fresh_base()

        if attack == 'zeros':
            malicious = model_zeros(fresh_base)

        elif attack == 'random':
            # Smart random: Gaussiano com sigma da camada (preserva mean/std globais)
            malicious = copy.deepcopy(fresh_base)
            for p in malicious.parameters():
                sigma = p.data.std().clamp_min(1e-8)
                p.data = torch.randn_like(p.data) * sigma

        elif attack == 'shuffle':
            # Shuffle parcial: permuta entre 30% e 100% dos pesos
            malicious = copy.deepcopy(fresh_base)
            frac = float(torch.empty(1).uniform_(0.3, 1.0))
            for p in malicious.parameters():
                flat = p.data.view(-1)
                n = int(len(flat) * frac)
                if n > 1:
                    idx = torch.randperm(len(flat))[:n]
                    perm = torch.randperm(n)
                    flat[idx] = flat[idx[perm]]

        elif attack == 'noise':
            # SNR uniform [3, 15] dB
            snr = float(torch.empty(1).uniform_(3.0, 15.0))
            malicious = model_noise(fresh_base, SNR=snr)
    else:
        # === Ataques originais (easy) ===
        # base_model compartilhado, ataques fixos
        malicious = create_malicious_model(base_model, attack_type=attack)

    state_dicts_list.append({
        'state_dict': malicious.state_dict(),
        'label': 1,
        'type': f'malicious_{attack}',
        'id': f'malicious_{attack}_{i:04d}',
    })

for entry in state_dicts_list:
    save_file(entry['state_dict'], f"state_dicts/{entry['id']}.safetensors")
    with open(f"state_dicts/{entry['id']}.json", 'w') as f:
        json.dump({'label': entry['label'], 'type': entry['type']}, f)

variant = ('Pretrained' if USE_PRETRAINED_BASE else 'Random-init') + (' + Hard' if HARDEN_ATTACKS else ' + Easy')
print(f'Salvos {len(state_dicts_list)} state_dicts em state_dicts/. Variante: {variant}')


### Salvamento do csv

In [24]:
import ast
import numpy as np
import pandas as pd

flattened_list = []
for entry in state_dicts_list:
    flattened_weights = {}
    shapes = {}
    for k, v in entry['state_dict'].items():
        if 'weight' in k:
            flattened_weights[k] = v.detach().cpu().numpy().flatten().tolist()
            shapes[f'shape_{k}'] = str(tuple(v.shape))
    flattened_list.append({
        'id': entry['id'],
        'label': entry['label'],
        'type': entry['type'],
        **flattened_weights,
        **shapes,
    })

df = pd.DataFrame(flattened_list)
df.to_csv('state_dicts.csv', index=False)


KeyboardInterrupt: 

### Leitura json e safetensors

In [ ]:
import glob
import json
from safetensors.torch import load_file
from datasets import Dataset

loaded_entries = []
for safetensor_file in sorted(glob.glob('state_dicts/*.safetensors')):
    sd = load_file(safetensor_file)
    json_file = safetensor_file.replace('.safetensors', '.json')
    with open(json_file, 'r') as f:
        meta = json.load(f)
    loaded_entries.append({
        'inputs': preprocess_weights(sd),
        'labels': meta['label'],
    })

assert loaded_entries, "Nenhum .safetensors em state_dicts/. Rode a celula de salvamento antes."
dataset = Dataset.from_list(loaded_entries)


### Leitura CSV

In [ ]:
df = pd.read_csv('state_dicts.csv')

loaded_entries = []
for _, row in df.iterrows():
    sd_reconstructed = {}
    for col in df.columns:
        if col.startswith('shape_'):  # Ignora shapes
            continue
        if col not in ['id', 'label', 'type']:  # Reconstrói apenas colunas de pesos
            shape_col = f'shape_{col}'  # Coluna correspondente
            if shape_col in row:
                shape_str = row[shape_col]
                shape = ast.literal_eval(shape_str)  # Parse '(32,1,5,5)' para tuple(32,1,5,5)
            else:
                raise ValueError(f"Shape não encontrado para {col}. Verifique o CSV.")
            
            if isinstance(row[col], str):  # Se salvo como string "[1.0, 2.0, ...]"
                values = ast.literal_eval(row[col])
            else:
                values = row[col]
            
            tensor = torch.tensor(values)  # Cria tensor 1D
            sd_reconstructed[col] = tensor.reshape(shape)  # Agora com shape correto!
    
    loaded_entries.append({
        'inputs': preprocess_weights(sd_reconstructed),
        'labels': row['label']
    })

dataset = Dataset.from_list(loaded_entries)

### Metodo completo para executar o treino

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load('accuracy')
f1 = evaluate.load('f1')
precision = evaluate.load('precision')
recall = evaluate.load('recall')


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy.compute(predictions=predictions, references=labels)['accuracy'],
        'f1': f1.compute(predictions=predictions, references=labels, average='binary')['f1'],
        'precision': precision.compute(predictions=predictions, references=labels, average='binary')['precision'],
        'recall': recall.compute(predictions=predictions, references=labels, average='binary')['recall'],
    }


In [ ]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model_name = 'distilbert-base-uncased'
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2, ignore_mismatched_sizes=True
)

lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=['q_lin', 'v_lin'])
model = get_peft_model(model, lora_config)

split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
tokenized_train = split_dataset['train'].map(
    tokenize_function, batched=True, remove_columns=['inputs']
)
tokenized_eval = split_dataset['test'].map(
    tokenize_function, batched=True, remove_columns=['inputs']
)

training_args = TrainingArguments(
    output_dir='./detector',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',
    logging_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to='none',
    metric_for_best_model='f1',
    greater_is_better=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)
trainer.train()


### Salvamento do modelo quantizado, alé mdisso fazer a leitura desse modelo e fazer a classificação do mesmo, para que possa ser usado para classificação no FL

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig

trainer.save_model('./detector_model')

if isinstance(model, PeftModel):
    model = model.merge_and_unload()
    model.save_pretrained('./detector_model_merged')
    model_path = './detector_model_merged'
else:
    model_path = './detector_model'

# 8-bit exige GPU; sem CUDA, usa fp32 padrao para nao quebrar.
if torch.cuda.is_available():
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0,
        llm_int8_has_fp16_weight=True,
    )
    quantized_model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        quantization_config=quantization_config,
        device_map='auto',
        torch_dtype=torch.float16,
    )
    quantized_model.save_pretrained('./detector_model_quantized_8bit')
    model = AutoModelForSequenceClassification.from_pretrained(
        './detector_model_quantized_8bit',
        device_map='auto',
    )
else:
    print('CUDA indisponivel; pulando quantizacao 8-bit e usando o modelo merged em CPU.')
    model = AutoModelForSequenceClassification.from_pretrained(model_path)


def classify_model(state_dict):
    inputs_list = preprocess_weights(state_dict)
    input_ids = torch.tensor([inputs_list], device=model.device)
    attention_mask = (input_ids != PAD_ID).long()
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    pred_label = torch.argmax(logits, dim=-1).item()
    classification = {0: 'benigno', 1: 'malicioso'}[pred_label]
    confidence = torch.softmax(logits, dim=-1)[0][pred_label].item()
    return classification, confidence


example_sd = create_malicious_model(base_model).state_dict()
label, conf = classify_model(example_sd)
print(f'Classificacao: {label} (confianca: {conf:.2f})')
